In [30]:
import numpy as np
import glob
import os
from array import array
import ROOT
import re

In [31]:
#Funzione per fit lineare 
def linear(x_data, m, q):
    return x_data*m + q

#Funzione per il calcolo dell'intercetta tra due fit lineari
def intercept_lin(m1, q1, m2, q2):
    intercept_x = (q2-q1)/(m1-m2)
    return intercept_x

In [35]:
    #Percentuali che definiscono il range 
    p_low= 0.25
    p_high= 0.50

In [36]:
 
# Il costrutto 'with' apre e chiude il file in automatico

filename="/Users/andrea/Desktop/INFN stage/Codici/Dati/ETL-PreSerie/HPK/QC-TS/Batch1/CV/NoIrr/PRE_HPK_QC-TS_W63_S5_LGAD-A[C-V Sweep - 2026-09-30 12_11_04 PM].txt"

In [37]:
#Creo il Canvas su cui disegnerò uno a uno i grafici e i fit di tutte le TS
    
canvas = ROOT.TCanvas("canvas", "1/C^2[V]", 800, 600)

#Estrazione di  W-S-LGAD
match = re.search(r'W(\d+).*?S(\d+).*?LGAD-([A-Z])', filename)        
if match:
    wafer = match.group(1)
    sensor = match.group(2)
    LGAD = match.group(3)
else:
    wafer = "NA" 
    sensor = "NA"
    LGAD = "NA"


canvas.Clear() #pulisco il canvas dai precedenti grafici e fit

with open(filename, "r") as file:
    Filelines = file.readlines()
    
try:
    #Tensione di inizio/fine, step in tensione e calcolo del numero di step nella misura
    Start_Voltage = float(Filelines[22].split()[1])
    Stop_Voltage = float(Filelines[23].split()[1])
    Step_Voltage = abs(float(Filelines[25].split()[1]))
    Num_biasStep = int((-Stop_Voltage+Start_Voltage)/Step_Voltage)
    
    #Print a terminale di tensione di inizio, fine e step
    #print("\nBias ramp:")
    #print("Start = " + str(Start_Voltage) + " V")
    #print("Stop= " + str(Stop_Voltage) + " V")
    #print("Step= " + str(Step_Voltage) + " V")
    
    #Filling delle liste con i valori dei parametri misurati
    Bias_Voltage = []
    Capacitance = []
    Capacitance2 = [] #1/C^2
    Line_first_data = 135 #Linea nel file di testo da cui iniziano i dati misurati di tensione e capacita'
    i = 0 #contatore del ciclo while successivo
    # questo ciclo while riempie le liste Bias_Voltage/I_pad con i dati misurati
    while i <= Num_biasStep:
        Bias_Voltage.append(abs(float(Filelines[Line_first_data + i].split()[0])))
        Capacitance.append(float(Filelines[Line_first_data + i].split()[1]))
        Capacitance2.append(1/(float(Filelines[Line_first_data + i].split()[1]))**2)
        i = i + 1


    #Creo un sistema per selezionare in automatico il range di fitting per la regone di bulk (La definiszione delle percentuali è all'inizio)
    C2_low = min(Capacitance2) + (max(Capacitance2)-min(Capacitance2))* p_low
    C2_high = min(Capacitance2) + (max(Capacitance2)-min(Capacitance2))* p_high
    closest_low = min(Capacitance2, key=lambda x: abs(x - C2_low))
    closest_high = min(Capacitance2, key=lambda x: abs(x - C2_high))
    #print(closest_low,closest_high)

    lower_C2_index = Capacitance2.index(closest_low)
    higher_C2_index = Capacitance2.index(closest_high)

    lower_V= Bias_Voltage[lower_C2_index]
    higher_V=Bias_Voltage[higher_C2_index]

    ### FIT gain layer region ###
    #limiti del fit
    low_GL_bias = 10.     #limite basso del fit (in voltaggio); deve essere un multiplo dello step in voltaggio
    high_GL_bias = 30.    #limite alto del fit (in voltaggio);deve essere un multiplo dello step in voltaggio
    i_low_GL = int(low_GL_bias/Step_Voltage)   #indice del limite basso del fit
    i_high_GL = int(high_GL_bias/Step_Voltage)    #indice del limite alto del fit
    
    i_low_bulk = lower_C2_index   #indice del limite basso del fit
    i_high_bulk = higher_C2_index   #indice del limite alto del fit
    #print(i_low_bulk,i_high_bulk)
    
    Bias_Voltage_GL = Bias_Voltage[i_low_GL:i_high_GL]
    Capacitance2_GL = Capacitance2[i_low_GL:i_high_GL]
    Bias_Voltage_bulk = Bias_Voltage[i_low_bulk:i_high_bulk]
    Capacitance2_bulk = Capacitance2[i_low_bulk:i_high_bulk]
        
    #Creo il grafico dei dati
    x_Data = array('d', Bias_Voltage)
    y_Data = array('d', Capacitance2)

    graph_Data = ROOT.TGraph(len(x_Data), x_Data, y_Data)
    graph_Data.SetMarkerStyle(20)      
    graph_Data.SetMarkerSize(0.3)
    graph_Data.SetMarkerColor(ROOT.kBlack)
    
    #Fit su GL
    fit_func_GL = ROOT.TF1("fit_func_GL", "[0]+[1]*x",Bias_Voltage_GL[0],Bias_Voltage_GL[-1]) 
    fit_func_GL.SetLineColor(ROOT.kRed)
    fit_func_GL.SetLineWidth(2)
    fit_func_GL.SetParameter(0,10**18)
    fit_func_GL.SetParameter(1,10**18)

    fit_GL = graph_Data.Fit(fit_func_GL, "RSQ") #R imposta il range relativo al fit, S salva e Q è la modalità quiet
    q_GL = fit_func_GL.GetParameter(0)
    m_GL = fit_func_GL.GetParameter(1)    

    
    #Fit sul bulk
    fit_func_bulk = ROOT.TF1("fit_func_bulk", "[0]+[1]*x",Bias_Voltage_bulk[0],Bias_Voltage_bulk[-1])
    fit_func_bulk.SetLineColor(ROOT.kGreen)
    fit_func_bulk.SetParameter(1,10**21)
    fit_func_bulk.SetParameter(0,-10**23)

    fit_bulk = graph_Data.Fit(fit_func_bulk, "RSQ+") #R imposta il range relativo al fit, S salva e Q è la modalità quiet, il + lo aggiunge allo stesso grafico
    q_bulk = fit_func_bulk.GetParameter(0)
    m_bulk = fit_func_bulk.GetParameter(1)
    
    #Stampo i risultati
    #print("\nGain Layer region *** Linear fit")
    #print("Angular coefficinet: m = " + str(m_GL))
    #print("y-intercept: q = " + str(q_GL))
    Covariance_matrix_GL = fit_GL.GetCovarianceMatrix()
    #print("sigma_m = " + str(Covariance_matrix_GL(1,1)))
    #print("sigma_q = " + str(Covariance_matrix_GL(0,0)))

    #print("\nBulk region *** Linear fit")
    #print(f"Il range di fitting della regione di bulk è: {Bias_Voltage_bulk[0]} - {Bias_Voltage_bulk[-1]}  ->  {len(Bias_Voltage_bulk)} punti")
    #print("Angular coefficinet: m = " + str(m_bulk))
    #print("y-intercept: q = " + str(q_bulk))
    Covariance_matrix_bulk = fit_bulk.GetCovarianceMatrix()
    #print("sigma_m = " + str(Covariance_matrix_bulk(1,1)))
    #print("sigma_q = " + str(Covariance_matrix_bulk(0,0)))

    #print("\n    ***   Depletion voltage of the gain layer   ***")
    V_GL=intercept_lin(m_GL, q_GL, m_bulk, q_bulk)
    #print("V_GL = " + str(V_GL) + " V\n\n")

    
    
    #Plot dei dati
    # Estendo il range di visibilità dei fit
    fit_func_GL.SetRange(Bias_Voltage[0], V_GL + 10)
    fit_func_bulk.SetRange(V_GL - 10, Bias_Voltage[-1])
    
    graph_Data.SetTitle(f"Sensor W{wafer} S{sensor} \nV_GL = {V_GL:.2f} V  ; Bias Voltage [V];1/C^{2} [1/F^{2}]")

    graph_Data.Draw("AP")
    graph_Data.SetMinimum(-10*10**21) 
    fit_func_GL.Draw("SAME")
    fit_func_bulk.Draw("SAME")

    canvas.Draw()
    canvas.Update()

    low_line = ROOT.TLine(lower_V, min(Capacitance2), lower_V, max(Capacitance2))
    low_line.SetLineColor(ROOT.kRed)
    low_line.SetLineStyle(2)
    low_line.Draw() 

    high_line = ROOT.TLine(higher_V, min(Capacitance2), higher_V, max(Capacitance2))
    high_line.SetLineColor(ROOT.kRed)
    high_line.SetLineStyle(2)
    high_line.Draw()

    ROOT.gStyle.SetOptStat(0)  # Nasconde il riquadro con Entries, Mean, Std Dev
    ROOT.gStyle.SetOptFit(0)   # Nasconde i parametri del fit (Chi2, Costante, Sigma...)

    # Creo legenda per il canvas
    legend = ROOT.TLegend(0.10, 0.75, 0.55, 0.88)
    legend.SetBorderSize(1)
    legend.SetFillColor(0)
    legend.SetTextSize(0.025)
    
    chi2_bulk = fit_func_bulk.GetChisquare()
    n_punti_bulk = len(Bias_Voltage_bulk)
    ndf_bulk = fit_func_bulk.GetNDF()
    
    chi2_red_bulk = chi2_bulk / ndf_bulk


    # Calcolo della distanza quadratica media (RMSE) rispetto al fit bulk
    residui_bulk = [
    y - (q_bulk + m_bulk * x)
    for x, y in zip(Bias_Voltage_bulk, Capacitance2_bulk)
    ]

    rmse_bulk = np.sqrt(np.mean(np.array(residui_bulk)**2))
    



    legend.AddEntry(
        fit_func_bulk,
        f"Bulk fit: #chi^{{2}}/NDF = {(chi2_red_bulk/10**40):.2f}, "
        f"N = {n_punti_bulk}, NDF = {ndf_bulk}",
        "l"
    )
    
    legend.Draw()
    
    canvas.Draw()
    png_name = filename.replace(".txt", ".png")
    canvas.SaveAs(png_name)

    print("\n---- FIT NELLA REGIONE DI BULK ------------------------")
    print(f"N di punti fit bulk = {n_punti_bulk:.2f}")
    print(f"Chi^2 ridotto per fit bulk = {chi2_red_bulk/10**40:.4f}")
    print(f"RMSE = {rmse_bulk/(10**21):.4f} * 10^21" )
    

            
except (IndexError, ValueError) as e:
    print(f" -> Errore di formattazione nel file {filename}: {e}")


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas
Info in <TCanvas::Print>: png file /Users/andrea/Desktop/INFN stage/Codici/Dati/ETL-PreSerie/HPK/QC-TS/Batch1/CV/NoIrr/PRE_HPK_QC-TS_W63_S5_LGAD-A[C-V Sweep - 2026-09-30 12_11_04 PM].png has been created



---- FIT NELLA REGIONE DI BULK ------------------------
N di punti fit bulk = 11.00
Chi^2 ridotto per fit bulk = 9.4194
RMSE = 0.2776 * 10^21
